# Logistic Regression News Topic Classification
## Arseniy Uspenskiy
## USPARS001

Importing necessary modules

In [5]:
# general imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import random 
import os
from typing import Optional

# pytorch
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

# scikit learn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, ConfusionMatrixDisplay

# constants
SPLITS = ("train", "dev", "test")
METHODS = ("bow", "tfidf")

method for cleaning text data

In [2]:
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"htpp\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

Class for loading text of a language

In [16]:
class NewsDataLoader:

    def __init__(self, root, language):
        self.root = root
        self.language = language
        self.directory = os.path.join(root, language)
        self.labels = self._load_labels()
        self.label2id = {lab: i for i, lab in enumerate(self.labels)}

    def _load_labels(self):
        path = os.path.join(self.directory, "labels.txt")
        with open(path, encoding="utf-8") as f:
            return [line.strip() for line in f if line.strip()]

    def _load_split(self, split):
        path = os.path.join(self.directory, f"{split}.tsv")
        df = pd.read_csv(path, sep="\t", engine="python")

        df["full_text"] = (df["headline"].fillna("") + " " + df["text"].fillna("")).apply(clean_text)
        df["label_id"] = df["category"].map(self.label2id)
        df["label_id"] = df["label_id"].astype(int)

        return df

    def load_splits(self):
        return {split: self._load_split(split) for split in SPLITS}

Class for extracting features for perceptron training

In [6]:
class FeatureExtractor:

    def __init__(self, method, max_features: Optional[int] = None, min_df: int = 1):
        self.method = method
        if method == "bow":
            self.vectorizer = CountVectorizer(max_features=max_features, min_df=min_df)
        else:
            self.vectorizer = TfidfVectorizer(max_features=max_features, min_df=min_df)
        
    def transform_train(self, train):
        return self.vectorizer.fit_transform(train)

    def transform(self, text):
        return self.vectorizer.transform(text)

    def vocab_size(self):
        return len(self.vectorizer.vocabulary_)

    def feature_names(self):
        return self.vectorizer.get_feature_names_out()

Method for using the two loader classes for building the full dataset of a language

In [13]:
def build_dataset(root, language, method, max_features: Optional[int] = None, min_df: int = 1):

    loader = NewsDataLoader(root, language)
    splits = loader.load_splits()
    extractor = FeatureExtractor(method, max_features=max_features, min_df=min_df)

    X_train = extractor.transform_train(splits["train"]["full_text"])
    X_val = extractor.transform(splits["dev"]["full_text"])
    X_test = extractor.transform(splits["test"]["full_text"])

    return X_train, X_val, X_test

def y_values(root, language):

    loader = NewsDataLoader(root, language)
    splits = loader.load_splits()

    return splits["train"]["label_id"].to_numpy(), splits["dev"]["label_id"].to_numpy(), splits["test"]["label_id"].to_numpy()

Load all language datasets

In [ ]:
eng_X_train_bow, eng_X_val_bow, eng_X_train_bow = build_dataset("news-dataset", "eng", "bow")
eng_X_train_tfidf, eng_X_val_tfidf, eng_X_train_tfidf = build_dataset("news-dataset", "eng", "tfidf")
eng_y_train, eng_y_val, eng_y_train = y_values("news-dataset", "eng")

sna_X_train_bow, sna_X_val_bow, sna_X_train_bow = build_dataset("news-dataset", "sna", "bow")
sna_X_train_tfidf, sna_X_val_tfidf, sna_X_train_tfidf = build_dataset("news-dataset", "sna", "tfidf")
sna_y_train, sna_y_val, sna_y_train = y_values("news-dataset", "sna")

xho_X_train_bow, xho_X_val_bow, xho_X_train_bow = build_dataset("news-dataset", "xho", "bow")
xho_X_train_tfidf, xho_X_val_tfidf, xho_X_train_tfidf = build_dataset("news-dataset", "xho", "tfidf")
xho_y_train, xho_y_val, xho_y_train = y_values("news-dataset", "xho")

TypeError: build_dataset() missing 1 required positional argument: 'method'

Multinomial logistic regression implementation

In [ ]:
# Perceptron

Create different feature extraction methods

In [ ]:
# extraction methods

Create hyperparameter testing arrays

In [ ]:
# creating tests

Training Perceptrons

In [ ]:
# Training code